In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
learning_rate = 1e-6
# Create random tensors as input and ground truth
x = torch.randn(64, 1000, device=device)  # input
y = torch.randn(64, 10, device=device)  # output

w1 = torch.randn(1000, 100, device=device)
w2 = torch.randn(100, 10, device=device)

# struct: relu(x*w1)*w2=y
for t in range(3000):
    h = x.mm(w1)  # X*W1
    h_relu = h.clamp(min=0)
    y_pred = h_relu.mm(w2)
    loss = y_pred - y

    grad_y_pred = 2.0 * loss
    grad_w2 = h_relu.t().mm(grad_y_pred)
    grad_h_relu = grad_y_pred.mm(w2.t())
    grad_h = grad_h_relu.clone()
    grad_h[h < 0] = 0
    grad_w1 = x.t().mm(grad_h)

    w1 -= learning_rate * grad_w1
    w2 -= learning_rate * grad_w2

    print(loss.pow(2).sum())

tensor(30714932., device='cuda:0')
tensor(22494572., device='cuda:0')
tensor(16520509., device='cuda:0')
tensor(11766514., device='cuda:0')
tensor(8124733., device='cuda:0')
tensor(5563603., device='cuda:0')
tensor(3856446., device='cuda:0')
tensor(2750719.7500, device='cuda:0')
tensor(2030667.2500, device='cuda:0')
tensor(1551746., device='cuda:0')
tensor(1222068., device='cuda:0')
tensor(986351.8750, device='cuda:0')
tensor(811682.6250, device='cuda:0')
tensor(678104.3125, device='cuda:0')
tensor(573229.2500, device='cuda:0')
tensor(489167.2500, device='cuda:0')
tensor(420652.7500, device='cuda:0')
tensor(364097.2500, device='cuda:0')
tensor(316926.9688, device='cuda:0')
tensor(277176.0625, device='cuda:0')
tensor(243437.4219, device='cuda:0')
tensor(214607.0469, device='cuda:0')
tensor(189843.9688, device='cuda:0')
tensor(168454.1562, device='cuda:0')
tensor(149886.9062, device='cuda:0')
tensor(133710.6406, device='cuda:0')
tensor(119594.7109, device='cuda:0')
tensor(107210.8438, de

這段是在「手刻反向傳播（backprop）」去算兩層網路的梯度。  
前向是：

$$
h = xW_1,\quad h_{relu}=\text{ReLU}(h),\quad y_{pred}=h_{relu}W_2
$$
也就是：
$$
x->w1->ReLu->w2->y
$$

損失你前面定義成 `loss = y_pred - y`，而實際最小化的是 $\sum (y_{pred}-y)^2$（你後面有 `loss.pow(2).sum()`）。

逐行解釋：

1. `grad_y_pred = 2.0 * loss`  
   這是  
   $$
   \frac{\partial}{\partial y_{pred}} \sum (y_{pred}-y)^2 = 2(y_{pred}-y)
   $$
   所以對輸出 `y_pred` 的梯度就是 `2*loss`。

2. `grad_w2 = h_relu.t().mm(grad_y_pred)`  
   因為 `y_pred = h_relu.mm(w2)`，對 `w2` 的梯度是  
   $$
   \frac{\partial L}{\partial W_2}=h_{relu}^T \frac{\partial L}{\partial y_{pred}}
   $$
   形狀也會對上：`(100,64) mm (64,10) -> (100,10)`，正好是 `w2` 的 shape。

3. `grad_h_relu = grad_y_pred.mm(w2.t())`  
   把梯度往前傳到 `h_relu`：  
   $$
   \frac{\partial L}{\partial h_{relu}}=\frac{\partial L}{\partial y_{pred}} W_2^T
   $$
   形狀：`(64,10) mm (10,100) -> (64,100)`。

4. `grad_h = grad_h_relu.clone()`  
   先複製一份梯度，避免直接改到原 tensor（後面要套 ReLU 的導數遮罩）。

5. `grad_h[h < 0] = 0`  
   ReLU 導數規則：  
   - $h > 0$ 時導數是 1  
   - $h < 0$ 時導數是 0  
   所以在 `h<0` 的位置，梯度要歸零，得到 $\frac{\partial L}{\partial h}$。

6. `grad_w1 = x.t().mm(grad_h)`  
   因為 `h = x.mm(w1)`，對 `w1` 的梯度是  
   $$
   \frac{\partial L}{\partial W_1}=x^T \frac{\partial L}{\partial h}
   $$
   形狀：`(1000,64) mm (64,100) -> (1000,100)`，正好是 `w1` 的 shape。

一句話總結：這 6 行就是從輸出層一路用 chain rule 往回傳，依序算出 `w2`、再穿過 ReLU、最後算 `w1` 的梯度。

**現在加上實際的dataset**


In [18]:
from torch.utils.data import TensorDataset, DataLoader

learning_rate = 1e-2
# Create random tensors as input and ground truth
x = torch.randn(64, 1000, device=device)  # input
y = torch.randn(64, 10, device=device)  # output

loader = DataLoader(TensorDataset(x, y), batch_size=8)


class TwoLayerNet(
    torch.nn.Module
):  # torch.nn: PyTorch 的神經網路模組（層、loss、Module 基底類別）
    def __init__(self, D_in, H, D_out):
        # super(): 呼叫父類別 nn.Module 的初始化，才能正確註冊參數給 optimizer
        super(TwoLayerNet, self).__init__()

        # Linear 是全連接層；常見還有 Conv2d、BatchNorm、Embedding、LSTM 等
        self.liner1 = torch.nn.Linear(D_in, H)
        self.liner2 = torch.nn.Linear(H, D_out)

    def forward(self, x):
        h = self.liner1(x)
        h_relu = torch.nn.functional.relu(h)
        y_pred = self.liner2(h_relu)
        return y_pred


modle = TwoLayerNet(D_in=1000, H=100, D_out=10)
modle = modle.to(device)  # .to(device): 把模型參數移到 CPU/GPU（要和 x、y 在同一裝置）

# parameters() 由 nn.Module 提供，會回傳所有可訓練參數（權重、偏置）
optimizer = torch.optim.Adam(modle.parameters(), lr=learning_rate)

for epochs in range(50):
    # for x_batch, y_batch in loader: DataLoader 每次迭代自動吐出一個 mini-batch
    for x_batch, y_batch in loader:
        y_pred = modle(x_batch)
        loss = torch.nn.functional.mse_loss(y_pred, y_batch)
        print("epoch", epochs, loss.item())

        loss.backward()  # 反向傳播：計算每個參數的梯度
        optimizer.step()  # 用目前梯度更新參數（真的改到模型權重）
        optimizer.zero_grad()  # 清空梯度，避免下一輪梯度累加

epoch 0 1.1158348321914673
epoch 0 1.1980743408203125
epoch 0 1.2979142665863037
epoch 0 1.344259262084961
epoch 0 1.1951836347579956
epoch 0 1.4502602815628052
epoch 0 0.9850377440452576
epoch 0 1.5814136266708374
epoch 1 13.048893928527832
epoch 1 9.399510383605957
epoch 1 3.5751640796661377
epoch 1 1.8807076215744019
epoch 1 1.758212685585022
epoch 1 1.3541539907455444
epoch 1 1.3134666681289673
epoch 1 0.829572856426239
epoch 2 8.248102188110352
epoch 2 3.656829833984375
epoch 2 2.0906870365142822
epoch 2 0.5530760884284973
epoch 2 0.5770798921585083
epoch 2 0.6730323433876038
epoch 2 0.5164212584495544
epoch 2 0.6921336054801941
epoch 3 2.135624408721924
epoch 3 1.1158316135406494
epoch 3 0.9542908072471619
epoch 3 0.43638715147972107
epoch 3 0.384957492351532
epoch 3 0.3910622298717499
epoch 3 0.29564881324768066
epoch 3 0.39357179403305054
epoch 4 0.5152637362480164
epoch 4 0.4802694320678711
epoch 4 0.317129522562027
epoch 4 0.2985103726387024
epoch 4 0.2215539962053299
epoch 4